In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# reading datasets about diabetes 

#df = pd.read_csv("/kaggle/input/diabetes-health-indicators-dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
df1 = pd.read_csv("/kaggle/input/diabetes-health-indicators-dataset/diabetes_binary_health_indicators_BRFSS2015.csv")
#df2 = pd.read_csv("/kaggle/input/diabetes-health-indicators-dataset/diabetes_012_health_indicators_BRFSS2015.csv")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import plotly.express as px
from imblearn.over_sampling import SMOTE
import os
import random
import tensorflow as tf
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, average_precision_score, 
                             classification_report, confusion_matrix, roc_auc_score, 
                             accuracy_score, precision_score, recall_score, f1_score)
from mlxtend.plotting import plot_confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras import regularizers
from tensorflow.keras.metrics import AUC
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import shap
import warnings

In [ ]:
df1.head()

In [ ]:
#checking shape of dataframe df1

df1.shape

# Checking duplicates row

In [ ]:
# 刪除重複值
duplicates = df1[df1.duplicated()]
print("Duplicate Rows : ",len(duplicates))
duplicates.head()

**result shows that there is 24206 duplicate rows in dataset df1**

In [ ]:
# eliminating 24206 duplicate rows from the dataset df1
df1.drop_duplicates(inplace = True)

In [ ]:
#checking shape after eliminating duplicate rows 
df1.shape

In [ ]:
# 1. 分割資料
#X = df1.drop('Diabetes_binary', axis=1)
#y = df1['Diabetes_binary']
#X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify = y, random_state=42)
#X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify = y_temp, random_state=42)
#
## 2. 對 BMI 進行對數轉換 (Log Transformation)
## 使用 np.log1p (log(1+x)) 較安全，可避免 BMI 為 0 時報錯
#X_train['BMI'] = np.log1p(X_train['BMI'])
#X_val['BMI'] = np.log1p(X_val['BMI'])
#X_test['BMI'] = np.log1p(X_test['BMI'])

# 使用NMI計算特徵關聯性

In [ ]:
from sklearn.metrics import normalized_mutual_info_score
# 參考論文：選擇可能影響目標的微觀/宏觀變數 [cite: 1052, 1128]
#check_features = ['BMI', 'MentHlth', 'PhysHlth', 'GenHlth', 'Age', 'Education', 'Income']
#
#def get_nmi_scores(X, y, features):
#    nmi_results = {}
#    for col in features:
#        # 計算特徵與目標的 NMI
#        # 論文指出 NMI 可量化觀察一項變數對減少另一項變數不確定性的程度 [cite: 1127]
#        score = normalized_mutual_info_score(X[col], y)
#        nmi_results[col] = score * 100  # 轉換為百分比 [%] 方便對照論文 [cite: 1128]
#    
#    # 排序 NMI 分數 [cite: 1128]
#    return pd.Series(nmi_results).sort_values(ascending=False)
#
#nmi_series = get_nmi_scores(X_train, y_train, check_features)
#print("特徵與 Diabetes_binary 的相關性 (NMI %):")
#print(nmi_series)

In [ ]:
#from sklearn.metrics import normalized_mutual_info_score

# 1. 取得 X_train 中的全部欄位名稱 [cite: 152]
# 這將包含所有原始特徵，確保沒有遺漏任何潛在的驅動力 [cite: 139, 142]
#all_features = X_train.columns.tolist()
#
## 2. 執行全體 NMI 計算
## 透過 NMI 量化每一個變數對減少目標變數不確定性的貢獻 [cite: 160, 161]
#all_nmi_scores = get_nmi_scores(X_train, y_train, all_features)
#
#print("全體特徵 NMI 排序 (%):")
#print(all_nmi_scores)
#
## 3. 根據論文標準 (如 NMI > 0.3%) 篩選出真正的顯著特徵 [cite: 167]
## 排除掉那些 NMI 極低、僅視為雜訊的變數 [cite: 168, 420]
#final_selected_features = all_nmi_scores[all_nmi_scores > 0.5].index.tolist()
#
#print(f"\n最終篩選出的顯著特徵總數: {len(final_selected_features)}")

實際使用NMI方法後無明顯幫助，因此最終無採用上述步驟

# TRAINING (Logistic Regression, SVM, ANN)

In [ ]:
# ==========================================
# 0. 設定全域隨機變數 (確保結果可重現)
# ==========================================
RANDOM_SEED = 42

os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ==========================================
# 1. 資料分割與前處理
# ==========================================
print(" 開始資料分割與前處理...")
# 假設 df1 已經存在環境中，請確認 df1 變數有效
X = df1.drop('Diabetes_binary', axis=1)
y = df1['Diabetes_binary']

# 1-1. 分割資料 (Train: 70%, Val: 15%, Test: 15%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_SEED)

# 使用 .copy() 避免 Pandas 的 SettingWithCopyWarning
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

# 轉換 y 為 numpy array (避免後續 KFold 切割 index 時發生 pandas 格式衝突)
y_train = y_train.values
y_val = y_val.values
y_test = y_test.values

# 1-2. 對 BMI 進行對數轉換 (Log Transformation)
X_train['BMI'] = np.log1p(X_train['BMI'])
X_val['BMI'] = np.log1p(X_val['BMI'])
X_test['BMI'] = np.log1p(X_test['BMI'])

# 1-3. 特徵縮放 (StandardScaler)
scaler = StandardScaler()
# 注意：fit 只能用在 X_train 上，這樣才能嚴格避免 Data Leakage
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print(" 資料分割與前處理完成！\n")

# ==========================================
# 2. 定義 ANN 模型結構
# ==========================================
def build_ann():
    model = Sequential()
    model.add(Dense(128, input_dim=X_train_scaled.shape[1], activation='relu', kernel_regularizer=regularizers.l2(0.0005)))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))
    
    model.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.0005)))
    model.add(Dropout(0.1))

    model.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.0005)))
    
    model.add(Dense(1, activation='sigmoid'))
    
    opt = Adam(learning_rate=0.0005)
    model.compile(loss='binary_crossentropy', optimizer=opt, metrics=[AUC(curve='ROC', name='auc'), AUC(curve='PR', name='pr_auc')])
    return model

# ==========================================
# 3. 一鍵訓練函數 (包含 5-Fold CV 與最終模型)
# ==========================================
def train_all_models(X_tr, y_tr, X_v, y_v):
    trained_models = {}
    cv_metrics = {'ANN': {'train': [], 'val': []}, 'Logistic': {'train': [], 'val': []}, 'SVM': {'train': [], 'val': []}}
    
    print(" 開始「5-Fold CV」與「最終模型訓練」流程...\n")
    
    # --- 執行 5-Fold Cross Validation ---
    print(" [階段 1/2] 正在執行 5-Fold Cross Validation (於 Train Set 內部進行)...")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_tr, y_tr), 1):
        print(f"   ▶ 執行 Fold {fold}/5 ...")
        X_f_tr, y_f_tr = X_tr[train_idx], y_tr[train_idx]
        X_f_v, y_f_v = X_tr[val_idx], y_tr[val_idx]
        
        # 1. ANN 訓練 (Fold)
        ann = build_ann()
        early_stop = EarlyStopping(monitor='val_auc', mode='max', patience=12, min_delta=0.001, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.3, patience=5, min_delta=0.001)
        ann.fit(X_f_tr, y_f_tr, validation_data=(X_f_v, y_f_v), epochs=500, batch_size=128, callbacks=[early_stop, reduce_lr], class_weight={0: 1.0, 1: 3.5}, verbose=0)
        cv_metrics['ANN']['train'].append(roc_auc_score(y_f_tr, ann.predict(X_f_tr, verbose=0).ravel()))
        cv_metrics['ANN']['val'].append(roc_auc_score(y_f_v, ann.predict(X_f_v, verbose=0).ravel()))

        # 2. Logistic Regression 訓練 (Fold)
        lr = LogisticRegression(class_weight={0: 1.0, 1: 7.0}, max_iter=4000, random_state=RANDOM_SEED)
        lr.fit(X_f_tr, y_f_tr)
        cv_metrics['Logistic']['train'].append(roc_auc_score(y_f_tr, lr.predict_proba(X_f_tr)[:, 1]))
        cv_metrics['Logistic']['val'].append(roc_auc_score(y_f_v, lr.predict_proba(X_f_v)[:, 1]))

        # 3. SVM 訓練與機率校準 (Fold) - 已修正為 base_estimator
        svm_base = LinearSVC(class_weight='balanced', max_iter=2000, dual=False, random_state=RANDOM_SEED)
        svm_base.fit(X_f_tr, y_f_tr)
        svm_calibrated = CalibratedClassifierCV(base_estimator=svm_base, cv='prefit')
        svm_calibrated.fit(X_f_v, y_f_v)
        cv_metrics['SVM']['train'].append(roc_auc_score(y_f_tr, svm_calibrated.predict_proba(X_f_tr)[:, 1]))
        cv_metrics['SVM']['val'].append(roc_auc_score(y_f_v, svm_calibrated.predict_proba(X_f_v)[:, 1]))

    # 計算並整理 CV 平均分數
    final_cv_metrics = {}
    print("\n 5-Fold CV 平均結果:")
    for name in cv_metrics:
        avg_tr = np.mean(cv_metrics[name]['train'])
        avg_v = np.mean(cv_metrics[name]['val'])
        final_cv_metrics[name] = {'train_auc': avg_tr, 'val_auc': avg_v}
        print(f"   - {name}: Train AUC = {avg_tr:.4f} | Validation AUC = {avg_v:.4f}")

    # --- 訓練最終模型 ---
    print("\n⏳ [階段 2/2] 正在使用 70% Train 訓練最終模型 (並以 15% Val 作為校準與 Early Stop)...")
    
    # 1. 最終 ANN
    ann_final = build_ann()
    early_stop_final = EarlyStopping(monitor='val_auc', mode='max', patience=12, min_delta=0.001, restore_best_weights=True)
    reduce_lr_final = ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.3, patience=5, min_delta=0.001)
    ann_final.fit(X_tr, y_tr, validation_data=(X_v, y_v), epochs=500, batch_size=128, callbacks=[early_stop_final, reduce_lr_final], class_weight={0: 1.0, 1: 3.5}, verbose=0)
    trained_models['ANN'] = ann_final
    
    # 2. 最終 Logistic
    lr_final = LogisticRegression(class_weight={0: 1.0, 1: 7.0}, max_iter=4000, random_state=RANDOM_SEED)
    lr_final.fit(X_tr, y_tr)
    trained_models['Logistic'] = lr_final
    
    # 3. 最終 SVM - 已修正為 base_estimator
    svm_base_final = LinearSVC(class_weight='balanced', max_iter=2000, dual=False, random_state=RANDOM_SEED)
    svm_base_final.fit(X_tr, y_tr)
    svm_calibrated_final = CalibratedClassifierCV(base_estimator=svm_base_final, cv='prefit')
    svm_calibrated_final.fit(X_v, y_v)
    trained_models['SVM'] = svm_calibrated_final

    print("🎉 所有模型訓練完畢！\n")
    return trained_models, final_cv_metrics

# ==========================================
# 4. 一鍵比較與繪圖函數 
# ==========================================
def compare_all_models(models_dict, X_tst, y_tst, thresholds=None):
    # 若未提供字典，預設全部使用 0.5
    if thresholds is None:
        thresholds = {name: 0.5 for name in models_dict.keys()}
        
    print("產出測試集圖表比較結果 ...\n")
    num_models = len(models_dict)
    fig = plt.figure(figsize=(18, 12))
    
    roc_ax = plt.subplot(2, 2, 1)
    pr_ax = plt.subplot(2, 2, 2)
    custom_preds = {}
    
    for name, model in models_dict.items()
        thresh = thresholds.get(name, 0.5) 
        
        if name == 'ANN':
            y_probs = model.predict(X_tst, verbose=0).ravel()
        else:
            y_probs = model.predict_proba(X_tst)[:, 1]
            
        y_pred_custom = (y_probs > thresh).astype("int32")
        custom_preds[name] = y_pred_custom
        
        fpr, tpr, _ = roc_curve(y_tst, y_probs)
        roc_auc_val = auc(fpr, tpr)
        roc_ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_val:.3f})')
        
        precision, recall, _ = precision_recall_curve(y_tst, y_probs)
        pr_auc_val = average_precision_score(y_tst, y_probs)
        pr_ax.plot(recall, precision, label=f'{name} (PR-AUC = {pr_auc_val:.3f})')

    roc_ax.plot([0, 1], [0, 1], 'k--')
    roc_ax.set_title('ROC Curve Comparison (Test Set)')
    roc_ax.set_xlabel('False Positive Rate')
    roc_ax.set_ylabel('True Positive Rate')
    roc_ax.legend(loc='lower right')
    
    pr_ax.set_title('Precision-Recall Curve Comparison (Test Set)')
    pr_ax.set_xlabel('Recall')
    pr_ax.set_ylabel('Precision')
    pr_ax.legend(loc='lower left')

    for idx, (name, y_pred_custom) in enumerate(custom_preds.items()):
        thresh = thresholds.get(name, 0.5)
        cm_ax = plt.subplot(2, num_models, num_models + 1 + idx)
        cm = confusion_matrix(y_tst, y_pred_custom)
        plot_confusion_matrix(conf_mat=cm, show_absolute=True, show_normed=True, class_names=['Healthy', 'Diabetes'], axis=cm_ax)
        cm_ax.set_title(f'{name} Confusion Matrix\n(Threshold > {thresh})')

    plt.tight_layout()
    plt.show()

    for name, y_pred_custom in custom_preds.items():
        thresh = thresholds.get(name, 0.5)
        print(f"\n{'='*25} {name} Test Report (Threshold = {thresh}) {'='*25}")
        print(classification_report(y_tst, y_pred_custom, target_names=['Healthy', 'Diabetes']))

# ==========================================
# 5. 整理各個模型的成效指標 (支援獨立 Threshold)
# ==========================================
def summarize_model_metrics(models_dict, X_tst, y_tst, thresholds=None):
    if thresholds is None:
        thresholds = {name: 0.5 for name in models_dict.keys()}
        
    print("\n📋 彙整各模型成效指標 (使用各模型自訂門檻值)...")
    results = []
    
    for name, model in models_dict.items():
        thresh = thresholds.get(name, 0.5)
        
        if name == 'ANN':
            y_probs = model.predict(X_tst, verbose=0).ravel()
        else:
            y_probs = model.predict_proba(X_tst)[:, 1]
            
        y_pred = (y_probs > thresh).astype(int)
        
        auc_score = roc_auc_score(y_tst, y_probs)
        acc = accuracy_score(y_tst, y_pred)
        prec = precision_score(y_tst, y_pred, zero_division=0)
        rec = recall_score(y_tst, y_pred)
        f1 = f1_score(y_tst, y_pred)
        
        results.append({
            'Model': name,
            'Threshold': thresh,   # 新增此欄位方便對照
            'AUC': round(auc_score, 4),
            'Accuracy': round(acc, 4),
            'Precision': round(prec, 4),
            'Recall': round(rec, 4),
            'F1-Score': round(f1, 4)
        })
        
    df_metrics = pd.DataFrame(results).set_index('Model')
    print("-" * 75)
    print(df_metrics.to_string())
    print("-" * 75)
    return df_metrics

# ==========================================
# 6. 過擬合檢查函數 
# ==========================================
def check_all_models_overfitting(models_dict, cv_metrics, X_tst, y_tst):
    print("\n🔍 進行所有模型的 Overfitting 檢查 (5-Fold CV Average vs Test)...\n")
    
    labels = list(models_dict.keys())
    train_aucs, val_aucs, test_aucs = [], [], []
    gaps = []
    
    for name, model in models_dict.items():
        tr_auc = cv_metrics[name]['train_auc']
        v_auc = cv_metrics[name]['val_auc']
        
        if name == 'ANN':
            y_test_probs = model.predict(X_tst, verbose=0).ravel()
        else:
            y_test_probs = model.predict_proba(X_tst)[:, 1]
            
        tst_auc = roc_auc_score(y_tst, y_test_probs)
        gap = tr_auc - v_auc
        
        train_aucs.append(tr_auc)
        val_aucs.append(v_auc)
        test_aucs.append(tst_auc)
        gaps.append(gap)

    x = np.arange(len(labels))
    width = 0.25  
    
    fig, ax = plt.subplots(figsize=(12, 7))
    rects1 = ax.bar(x - width, train_aucs, width, label='CV Train AUC (Avg)', color='#ecf0f1', edgecolor='#7f8c8d', alpha=0.9)
    rects2 = ax.bar(x, val_aucs, width, label='CV Validation AUC (Avg)', color='#3498db', alpha=0.9)
    rects3 = ax.bar(x + width, test_aucs, width, label='Test AUC', color='#2980b9', alpha=0.9)
    
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', 
                        fontsize=9, fontweight='bold')

    autolabel(rects1)
    autolabel(rects2)
    autolabel(rects3)

    for i, gap in enumerate(gaps):
        color = '#c0392b' if gap > 0.05 else '#27ae60'
        ax.text(i, 0.05, f'CV Train-Val Gap:\n{gap:.4f}', ha='center', va='bottom', 
                color='white', fontweight='bold', bbox=dict(facecolor=color, alpha=0.8, edgecolor='none', boxstyle='round,pad=0.5'))

    ax.set_ylabel('ROC-AUC Score')
    ax.set_title('Overfitting Check: CV Train vs CV Validation vs Test AUC', fontsize=14, pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontweight='bold', fontsize=12)
    ax.set_ylim(0, 1.15) 
    ax.legend(loc='upper right')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    
    plt.tight_layout()
    plt.show()

print('---------------------------------------------------')
# ==========================================
# 7. 最終主執行區塊
# ==========================================
# 1. 執行 5-Fold 訓練
my_models, my_cv_metrics = train_all_models(X_train_scaled, y_train, X_val_scaled, y_val)

# --- 定義各模型的專屬 Threshold ---
custom_thresholds = {
    'ANN': 0.3,
    'Logistic': 0.35,  # 假設 Logistic 適合 0.35
    'SVM': 0.4         # 假設 SVM 適合 0.4
}

# 2. 產出測試集圖表比較結果 (傳入 custom_thresholds)
compare_all_models(my_models, X_test_scaled, y_test, thresholds=custom_thresholds)

# 3. 產出測試集成效指標 DataFrame (傳入 custom_thresholds)
metrics_table = summarize_model_metrics(my_models, X_test_scaled, y_test, thresholds=custom_thresholds)

# 4. 檢查過擬合 (維持不變，因為 AUC 計算不依賴門檻值)
check_all_models_overfitting(my_models, my_cv_metrics, X_test_scaled, y_test)

# 調整模型Threshold

In [ ]:
# --- 定義各模型的專屬 Threshold ---
custom_thresholds = {
    'ANN': 0.3,
    'Logistic': 0.4,  
    'SVM': 0.10        
}

# 2. 產出測試集圖表比較結果 
compare_all_models(my_models, X_test_scaled, y_test, thresholds=custom_thresholds)

# 3. 產出測試集成效指標 DataFrame
metrics_table = summarize_model_metrics(my_models, X_test_scaled, y_test, thresholds=custom_thresholds)

# 4. 檢查過擬合 
check_all_models_overfitting(my_models, my_cv_metrics, X_test_scaled, y_test)

# SHAP 模型解釋 

In [ ]:
# ==========================================
# 6. SHAP 模型解釋 
# ==========================================
def explain_ann_with_shap_v2(model, X_train_scaled, X_test_scaled, feature_names):
    warnings.filterwarnings("ignore", message="The default of 'normalize' will be set to False")
    warnings.filterwarnings("ignore", message="Set parameter alpha to")
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    print("\n 正在計算 SHAP ")
    
    # 封裝預測函數
    def model_predict(data):
        return model.predict(data, verbose=0).flatten()

    # 建立背景集 (從訓練集取 100 筆縮放後的資料)
    background = X_train_scaled[np.random.choice(X_train_scaled.shape[0], 100, replace=False)]
    
    # 使用 KernelExplainer
    # link="logit" 適合處理 binary classification 的機率輸出
    explainer = shap.KernelExplainer(model_predict, background, link="identity")
    
    # 選取測試集的前 30 筆 
    test_samples = X_test_scaled[:30]
    
    # 計算 SHAP 值 
    shap_values = explainer.shap_values(test_samples)

    # 3. 視覺化 - Summary Plot
    plt.figure(figsize=(10, 6))
    plt.title("SHAP Summary Plot (ANN - Overfitting Checked)")
    shap.summary_plot(shap_values, test_samples, feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.show()

    # 4. 視覺化 - Waterfall Plot (針對第一筆資料)
    # 建立 Explanation 物件以符合新版繪圖 API
    exp = shap.Explanation(
        values=shap_values, 
        base_values=explainer.expected_value, 
        data=test_samples, 
        feature_names=feature_names
    )
    
    print("\n 正在產生單一樣本的 Waterfall Plot (第 1 筆測試資料)...")
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(exp[0], show=False)
    plt.show()

# ==========================================
# 執行
# ==========================================
feature_list = X_train.columns.tolist() if hasattr(X_train, 'columns') else [f"F_{i}" for i in range(X_train_scaled.shape[1])]
explain_ann_with_shap_v2(my_models['ANN'], X_train_scaled, X_test_scaled, feature_list)

# 單一 True Positive 個案分析

In [ ]:
# ==========================================
# 7. 單一 True Positive 個案分析 (ANN)
# ==========================================
def explain_specific_tp_case(model, X_tst_scaled, y_tst, feature_names, threshold=0.3):
    """
    自動尋找 True Positive (預測為1且實際為1) 的案例並進行 SHAP 解釋
    """
    print(f"\n 正在搜尋 True Positive 案例 (Threshold > {threshold})...")
    
    # 1. 取得模型預測機率
    y_probs = model.predict(X_tst_scaled, verbose=0).ravel()
    y_pred = (y_probs > threshold).astype(int)
    
    # 2. 找出 True Positive 的索引 (實際是1 且 預測也是1)
    # y_tst 如果是 Series 或 DataFrame，需轉為 numpy array 方便比較
    y_tst_arr = np.array(y_tst).flatten()
    tp_indices = np.where((y_tst_arr == 1) & (y_pred == 1))[0]
    
    if len(tp_indices) == 0:
        print("找不到 True Positive 案例")
        return

    # 隨機挑選一個 TP 案例 (或者你可以固定選第一個 tp_indices[0])
    chosen_idx = tp_indices[0] 
    print(f"✅ 找到 {len(tp_indices)} 個符合條件的案例，選取測試集索引: {chosen_idx}")
    print(f"   - 實際標籤 (Actual): Diabetes (1)")
    print(f"   - 預測機率 (Probability): {y_probs[chosen_idx]:.4f}")

    # 3. 準備 SHAP 解釋器 (使用小規模背景集以維持速度)
    # 這裡我們直接使用 KernelExplainer 的簡化版邏輯
    def model_predict(data):
        return model.predict(data, verbose=0).flatten()

    # 隨機取 100 筆訓練資料當背景 (假設 X_train_scaled 已定義)
    background = X_train_scaled[np.random.choice(X_train_scaled.shape[0], 100, replace=False)]
    explainer = shap.KernelExplainer(model_predict, background)
    
    # 4. 計算該特定個案的 SHAP 值
    # 注意：shap_values 會是一個 list 或 array，取對應的部分
    single_sample = X_tst_scaled[chosen_idx : chosen_idx + 1]
    shap_val_single = explainer.shap_values(single_sample)
    
    if isinstance(shap_val_single, list):
        shap_val_single = shap_val_single[0]

    # 5. 繪製 Waterfall Plot
    # 建立 Explanation 物件
    exp_single = shap.Explanation(
        values=np.squeeze(shap_val_single), 
        base_values=explainer.expected_value, 
        data=np.squeeze(single_sample), 
        feature_names=feature_names
    )
    
    plt.figure(figsize=(12, 8))
    plt.title(f"SHAP Analysis for True Positive Case (Index: {chosen_idx})", fontsize=14)
    shap.plots.waterfall(exp_single, show=False)
    plt.show()

# ==========================================
# 執行分析
# ==========================================
# 確保 feature_list 已經定義
feature_list = X_train.columns.tolist() if hasattr(X_train, 'columns') else [f"F_{i}" for i in range(X_train_scaled.shape[1])]

explain_specific_tp_case(my_models['ANN'], X_test_scaled, y_test, feature_list, threshold=0.3)